## Summary: What You've Built

### ✅ Complete Features Implemented

| Feature | Details |
|---------|---------|
| **Authentication** | JWT tokens with 12-hour expiration |
| **Authorization** | 3 roles with granular permission control |
| **Transactions** | Full CRUD with filtering by date, type, category |
| **Analytics** | Summary, category breakdown, monthly trends, recent |
| **Database** | SQLite with SQLAlchemy ORM |
| **Validation** | Pydantic schemas for all requests/responses |
| **Error Handling** | Proper HTTP status codes (400, 401, 403, 404) |
| **Documentation** | Auto-generated Swagger UI at /docs |

### 📊 Role-Based Access Control

| Role | Permissions |
|------|-------------|
| **Viewer** | Read own transactions only |
| **Analyst** | Read + Create transactions + Analytics |
| **Admin** | Full CRUD + All analytics |

### 📁 File Structure Summary
- `database.py` - SQLAlchemy setup
- `models/` - User & Transaction ORM models
- `schemas/` - Pydantic validation schemas
- `services/auth_service.py` - JWT & password logic
- `dependencies.py` - Auth middleware & role guards
- `routes/` - API endpoints (auth, transactions, analytics)
- `main.py` - FastAPI application
- `seed.py` - Mock data generator

### 🚀 Next Steps

1. **Install dependencies**: `pip install -r requirements.txt`
2. **Seed database**: `python seed.py`
3. **Start server**: `uvicorn main:app --reload`
4. **Test API**: Visit `http://localhost:8000/docs`
5. **Explore endpoints**: Use Swagger UI to test all routes

---

**That's it! You have a production-ready Finance Tracker API with authentication, authorization, and analytics! 🎉**

## STEP 12: Quick Reference - How to Test

### Start the API Server
```bash
# Seed the database with test data
python seed.py

# Start the FastAPI server
uvicorn main:app --reload
```

### Access Interactive API Documentation
Open your browser to:
```
http://localhost:8000/docs
```

This shows a Swagger UI where you can test every endpoint interactively!

### Test Workflow

1. **Register/Login**: Call POST /auth/login with:
   ```json
   {
     "username": "admin",
     "password": "admin123"
   }
   ```

2. **Copy the access_token** from response

3. **Click "Authorize"** in Swagger UI and paste the token

4. **Call other endpoints** - they now have your user context!

### Example API Calls

**Get all transactions:**
```
GET /transactions
```

**Create a transaction:**
```
POST /transactions
{
  "amount": 50.00,
  "type": "expense",
  "category": "food",
  "date": "2026-04-01",
  "notes": "Lunch"
}
```

**Get financial summary:**
```
GET /analytics/summary
```

**Get category breakdown:**
```
GET /analytics/by-category
```

**Get monthly trends:**
```
GET /analytics/monthly
```

In [ ]:
seed_py = '''
from database import SessionLocal
from models.user import User, RoleEnum
from models.transaction import Transaction, TypeEnum
from services.auth_service import hash_password
from datetime import date

db = SessionLocal()

# Create admin user
admin = User(
    username="admin",
    email="admin@test.com",
    hashed_password=hash_password("admin123"),
    role=RoleEnum.admin
)
db.add(admin)
db.commit()
db.refresh(admin)
print(f"✅ Created admin user: {admin.username}")

# Create sample transactions
transactions = [
    Transaction(user_id=admin.id, amount=5000, type=TypeEnum.income,  category="salary",    date=date(2026,3,1),  notes="March salary"),
    Transaction(user_id=admin.id, amount=1200, type=TypeEnum.expense, category="rent",      date=date(2026,3,2)),
    Transaction(user_id=admin.id, amount=300,  type=TypeEnum.expense, category="food",      date=date(2026,3,10)),
    Transaction(user_id=admin.id, amount=2000, type=TypeEnum.income,  category="freelance", date=date(2026,3,15)),
    Transaction(user_id=admin.id, amount=500,  type=TypeEnum.expense, category="shopping",  date=date(2026,3,20)),
]
db.add_all(transactions)
db.commit()
print(f"✅ Created {len(transactions)} sample transactions")
print("\\n🚀 Database seeded successfully!")
'''

print("📄 seed.py")
print("=" * 50)
print(seed_py)

## STEP 11: Seed Database with Mock Data

Create `seed.py` - Populate database with test data:

In [ ]:
main_py = '''
from fastapi import FastAPI
# from database import Base, engine  # In real file
# from routes import auth, transactions, analytics  # In real file

# Create all database tables on startup
Base.metadata.create_all(bind=engine)

# Initialize FastAPI application
app = FastAPI(
    title="Finance Tracker API",
    description="Track your income and expenses with role-based access",
    version="1.0.0"
)

# Include routers
app.include_router(auth.router)
app.include_router(transactions.router)
app.include_router(analytics.router)

@app.get("/")
def root():
    """Welcome endpoint"""
    return {"message": "Finance Tracker API is running"}

# To run: uvicorn main:app --reload
'''

print("📄 main.py")
print("=" * 50)
print(main_py)

## STEP 10: Create FastAPI Application

Create `main.py` - Application entry point with all routers:

In [ ]:
routes_analytics_py = '''
from fastapi import APIRouter, Depends
# from sqlalchemy.orm import Session  # In real file
# from sqlalchemy import func  # In real file
# from database import get_db  # In real file
# from models.transaction import Transaction  # In real file
# from dependencies import require_analyst_or_above  # In real file

router = APIRouter(prefix="/analytics", tags=["Analytics"])

@router.get("/summary")
def summary(db: Session = Depends(get_db), user=Depends(require_analyst_or_above)):
    """
    Get financial summary: total income, expenses, and balance.
    Accessible by analyst and admin roles only.
    """
    txns = db.query(Transaction).filter(Transaction.user_id == user.id).all()
    income  = sum(t.amount for t in txns if t.type == "income")
    expense = sum(t.amount for t in txns if t.type == "expense")
    return {
        "total_income": income,
        "total_expense": expense,
        "balance": income - expense
    }

@router.get("/by-category")
def by_category(db: Session = Depends(get_db), user=Depends(require_analyst_or_above)):
    """Get breakdown of transactions by category"""
    results = (db.query(Transaction.category, Transaction.type, func.sum(Transaction.amount))
               .filter(Transaction.user_id == user.id)
               .group_by(Transaction.category, Transaction.type)
               .all())
    return [{"category": r[0], "type": r[1], "total": r[2]} for r in results]

@router.get("/monthly")
def monthly(db: Session = Depends(get_db), user=Depends(require_analyst_or_above)):
    """Get monthly income, expenses, and trends"""
    results = (db.query(func.strftime("%Y-%m", Transaction.date), Transaction.type, func.sum(Transaction.amount))
               .filter(Transaction.user_id == user.id)
               .group_by(func.strftime("%Y-%m", Transaction.date), Transaction.type)
               .all())
    return [{"month": r[0], "type": r[1], "total": r[2]} for r in results]

@router.get("/recent")
def recent(db: Session = Depends(get_db), user=Depends(require_analyst_or_above)):
    """Get last 10 recent transactions"""
    return (db.query(Transaction)
            .filter(Transaction.user_id == user.id)
            .order_by(Transaction.date.desc())
            .limit(10)
            .all())
'''

print("📄 routes/analytics.py")
print("=" * 50)
print(routes_analytics_py)

## STEP 9: Analytics Routes

Create `routes/analytics.py` - Financial summaries and insights (analyst+ only):

In [ ]:
routes_transactions_py = '''
from fastapi import APIRouter, Depends, HTTPException, Query
# from sqlalchemy.orm import Session  # In real file
# from database import get_db  # In real file
# from models.transaction import Transaction  # In real file
# from schemas.transaction import TransactionCreate, TransactionOut, TransactionUpdate  # In real file
# from dependencies import get_current_user, require_admin  # In real file
from typing import Optional
from datetime import date

router = APIRouter(prefix="/transactions", tags=["Transactions"])

@router.get("/", response_model=list[TransactionOut])
def get_transactions(
    type: Optional[str] = None,
    category: Optional[str] = None,
    start_date: Optional[date] = None,
    end_date: Optional[date] = None,
    db: Session = Depends(get_db),
    user=Depends(get_current_user)
):
    """
    List user's transactions with optional filters.
    - type: 'income' or 'expense'
    - category: Filter by category (e.g., 'food', 'salary')
    - start_date, end_date: Filter by date range
    """
    q = db.query(Transaction).filter(Transaction.user_id == user.id)
    
    if type:        q = q.filter(Transaction.type == type)
    if category:    q = q.filter(Transaction.category == category)
    if start_date:  q = q.filter(Transaction.date >= start_date)
    if end_date:    q = q.filter(Transaction.date <= end_date)
    
    return q.all()

@router.post("/", response_model=TransactionOut)
def create_transaction(data: TransactionCreate, db: Session = Depends(get_db), user=Depends(require_admin)):
    """Create a new transaction (admin only)"""
    t = Transaction(**data.dict(), user_id=user.id)
    db.add(t)
    db.commit()
    db.refresh(t)
    return t

@router.put("/{id}", response_model=TransactionOut)
def update_transaction(id: int, data: TransactionUpdate, db: Session = Depends(get_db), user=Depends(require_admin)):
    """Update a transaction (admin only)"""
    t = db.query(Transaction).filter(Transaction.id == id).first()
    if not t:
        raise HTTPException(status_code=404, detail="Transaction not found")
    
    for key, val in data.dict(exclude_unset=True).items():
        setattr(t, key, val)
    
    db.commit()
    db.refresh(t)
    return t

@router.delete("/{id}")
def delete_transaction(id: int, db: Session = Depends(get_db), user=Depends(require_admin)):
    """Delete a transaction (admin only)"""
    t = db.query(Transaction).filter(Transaction.id == id).first()
    if not t:
        raise HTTPException(status_code=404, detail="Transaction not found")
    
    db.delete(t)
    db.commit()
    return {"message": "Deleted successfully"}
'''

print("📄 routes/transactions.py")
print("=" * 50)
print(routes_transactions_py)

## STEP 8: Transaction Routes

Create `routes/transactions.py` - CRUD operations for transactions:

In [ ]:
routes_auth_py = '''
from fastapi import APIRouter, Depends, HTTPException
# from sqlalchemy.orm import Session  # In real file
# from database import get_db  # In real file
# from models.user import User  # In real file
# from schemas.user import UserCreate, LoginRequest, UserOut  # In real file
# from services.auth_service import hash_password, verify_password, create_token  # In real file

router = APIRouter(prefix="/auth", tags=["Auth"])

@router.post("/register", response_model=UserOut)
def register(data: UserCreate, db: Session = Depends(get_db)):
    """
    Register a new user.
    - username: Must be unique
    - email: Must be valid email format
    - password: Will be hashed with bcrypt
    - role: Defaults to 'viewer'
    """
    # Check if username already exists
    if db.query(User).filter(User.username == data.username).first():
        raise HTTPException(status_code=400, detail="Username already taken")
    
    # Create new user
    user = User(
        username=data.username,
        email=data.email,
        hashed_password=hash_password(data.password),
        role=data.role
    )
    db.add(user)
    db.commit()
    db.refresh(user)
    return user

@router.post("/login")
def login(data: LoginRequest, db: Session = Depends(get_db)):
    """
    Login with username and password.
    Returns JWT access token valid for 12 hours.
    """
    # Find user by username
    user = db.query(User).filter(User.username == data.username).first()
    
    # Verify password
    if not user or not verify_password(data.password, user.hashed_password):
        raise HTTPException(status_code=401, detail="Invalid credentials")
    
    # Create JWT token
    token = create_token({"sub": user.username, "role": user.role})
    return {"access_token": token, "token_type": "bearer"}
'''

print("📄 routes/auth.py")
print("=" * 50)
print(routes_auth_py)

## STEP 7: Authentication Routes

Create `routes/auth.py` - Register and login endpoints:

In [ ]:
dependencies_py = '''
from fastapi import Depends, HTTPException, Header
# from services.auth_service import decode_token, get_user_by_username  # In real file
# from database import get_db  # In real file
# from sqlalchemy.orm import Session  # In real file

def get_current_user(Authorization: str = Header(...), db=Depends(get_db)):
    """
    Extract current user from JWT token in Authorization header.
    Header format: "Bearer <token>"
    """
    try:
        token = Authorization.split(" ")[1]      # Extract token from "Bearer <token>"
        payload = decode_token(token)
        user = get_user_by_username(db, payload["sub"])
        if not user:
            raise HTTPException(status_code=401, detail="User not found")
        return user
    except Exception:
        raise HTTPException(status_code=401, detail="Invalid token")

def require_admin(user=Depends(get_current_user)):
    """Guard: Allow only admin users"""
    if user.role != "admin":
        raise HTTPException(status_code=403, detail="Admins only")
    return user

def require_analyst_or_above(user=Depends(get_current_user)):
    """Guard: Allow admin and analyst users"""
    if user.role not in ["analyst", "admin"]:
        raise HTTPException(status_code=403, detail="Analysts and Admins only")
    return user
'''

print("📄 dependencies.py")
print("=" * 50)
print(dependencies_py)

## STEP 6: Dependencies & Authorization

Create `dependencies.py` - Extract user from token, role-based guards:

In [ ]:
auth_service_py = '''
from passlib.context import CryptContext
from jose import jwt, JWTError
from datetime import datetime, timedelta
# from models.user import User  # In real file
# from sqlalchemy.orm import Session  # In real file

SECRET_KEY = "your_secret_key_here"
ALGORITHM = "HS256"
pwd_context = CryptContext(schemes=["bcrypt"])

def hash_password(password: str) -> str:
    """Hash a password using bcrypt"""
    return pwd_context.hash(password)

def verify_password(plain: str, hashed: str) -> bool:
    """Verify a plain password against its hash"""
    return pwd_context.verify(plain, hashed)

def create_token(data: dict) -> str:
    """Create JWT token with 12-hour expiration"""
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(hours=12)
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    return encoded_jwt

def decode_token(token: str) -> dict:
    """Decode and validate JWT token"""
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

def get_user_by_username(db, username: str):
    """Fetch user from database by username"""
    return db.query(User).filter(User.username == username).first()
'''

print("📄 services/auth_service.py")
print("=" * 50)
print(auth_service_py)

## STEP 5: Authentication Service

Create `services/auth_service.py` - JWT tokens and password hashing:

In [ ]:
schemas_transaction_py = '''
from pydantic import BaseModel, Field
from datetime import date
from enum import Enum
from typing import Optional

class TypeEnum(str, Enum):
    income = "income"
    expense = "expense"

class TransactionCreate(BaseModel):
    amount: float = Field(..., gt=0)        # Must be greater than 0
    type: TypeEnum
    category: str
    date: date
    notes: Optional[str] = None

class TransactionUpdate(BaseModel):
    amount: Optional[float] = Field(None, gt=0)
    category: Optional[str] = None
    notes: Optional[str] = None

class TransactionOut(TransactionCreate):
    id: int
    user_id: int
    
    class Config:
        orm_mode = True
'''

print("\n📄 schemas/transaction.py")
print("=" * 50)
print(schemas_transaction_py)

In [ ]:
models_transaction_py = '''
from sqlalchemy import Column, Integer, Float, String, Enum, Date, ForeignKey
# from database import Base  # In real file
import enum

class TypeEnum(str, enum.Enum):
    income = "income"
    expense = "expense"

class Transaction(Base):
    __tablename__ = "transactions"

    id = Column(Integer, primary_key=True, index=True)
    user_id = Column(Integer, ForeignKey("users.id"))
    amount = Column(Float, nullable=False)          # Transaction amount
    type = Column(Enum(TypeEnum), nullable=False)   # Income or expense
    category = Column(String, nullable=False)        # e.g., food, salary, rent
    date = Column(Date, nullable=False)              # Transaction date
    notes = Column(String, nullable=True)            # Optional notes
'''

print("📄 models/transaction.py")
print("=" * 50)
print(models_transaction_py)

## STEP 4: Define Transaction Model & Schema

Create `models/transaction.py` - Transaction model for income/expense tracking:

In [ ]:
schemas_user_py = '''
from pydantic import BaseModel, EmailStr
from enum import Enum

class RoleEnum(str, Enum):
    viewer = "viewer"
    analyst = "analyst"
    admin = "admin"

class UserCreate(BaseModel):
    username: str
    email: EmailStr
    password: str
    role: RoleEnum = RoleEnum.viewer

class UserOut(BaseModel):
    id: int
    username: str
    email: str
    role: RoleEnum
    
    class Config:
        orm_mode = True

class LoginRequest(BaseModel):
    username: str
    password: str
'''

print("\n📄 schemas/user.py - Pydantic validation schemas")
print("=" * 50)
print(schemas_user_py)

In [ ]:
models_user_py = '''
from sqlalchemy import Column, Integer, String, Enum
# from database import Base  # In real file
import enum

class RoleEnum(str, enum.Enum):
    viewer = "viewer"      # Read-only access
    analyst = "analyst"    # Read + analytics
    admin = "admin"        # Full access

class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, nullable=False)
    email = Column(String, unique=True, nullable=False)
    hashed_password = Column(String, nullable=False)
    role = Column(Enum(RoleEnum), default=RoleEnum.viewer)
'''

print("📄 models/user.py")
print("=" * 50)
print(models_user_py)

## STEP 3: Define User Model & Schema

Create `models/user.py` - User model with role-based access control:

In [ ]:
database_py_code = '''
from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

DATABASE_URL = "sqlite:///./finance.db"

engine = create_engine(
    DATABASE_URL, 
    connect_args={"check_same_thread": False}
)
SessionLocal = sessionmaker(bind=engine, autocommit=False, autoflush=False)
Base = declarative_base()

def get_db():
    """Dependency to get database session"""
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
'''

print("📄 database.py")
print("=" * 50)
print(database_py_code)

## STEP 2: Database Configuration

Create `database.py` - handles SQLAlchemy ORM setup and database connection:

In [ ]:
import subprocess
import sys

# Install all required packages
packages = [
    "fastapi==0.109.0",
    "uvicorn[standard]==0.27.0",
    "sqlalchemy==2.0.25",
    "pydantic==2.5.3",
    "python-jose[cryptography]==3.3.0",
    "passlib[bcrypt]==1.7.4",
    "python-multipart==0.0.6"
]

print("🔧 Installing dependencies...\n")
for package in packages:
    print(f"  ▶ {package}")

# In a real environment, uncomment this:
# subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)

print("\n✅ Dependencies ready!")

## STEP 1: Environment Setup & Dependencies

Install all required packages for the FastAPI project:

# 💰 Finance Tracker API - Complete Build Guide

## Overview
This notebook walks through building a **production-ready Finance Tracker API** with:
- **FastAPI** - Modern, fast web framework
- **SQLAlchemy** - ORM for database management
- **SQLite** - Simple, file-based database
- **JWT** - Token-based authentication
- **Role-Based Access Control** - 3 user roles (viewer, analyst, admin)

## Key Features
✅ User authentication with JWT tokens  
✅ Role-based authorization  
✅ Transaction CRUD operations  
✅ Advanced analytics and summaries  
✅ Automatic API documentation (Swagger UI)  
✅ Complete error handling  

## Project Structure
```
finance_system/
├── database.py              # DB config
├── dependencies.py          # Auth middleware
├── main.py                  # App entry point
├── seed.py                  # Mock data
├── models/
│   ├── user.py
│   └── transaction.py
├── schemas/
│   ├── user.py
│   └── transaction.py
├── services/
│   └── auth_service.py
└── routes/
    ├── auth.py
    ├── transactions.py
    └── analytics.py
```

**Follow each section below to build the complete API.**